## IMPORTAR LAS LIBRERIAS

Actualizar a las que se usen finalmente en tu proyecto.

In [26]:
import numpy as np
import pandas as pd
import cloudpickle

#Automcompletar rapido
%config IPCompleter.greedy=True

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

## CARGAR LOS DATOS

In [2]:
ruta_proyecto = 'C:/Users/vifra/vidal cientifico de datos/Portafolio ML/Proyecto leadscoring/00_PROYECTO_LEADSCORING/'

In [3]:
nombre_fichero_datos = 'Leads.csv'

In [4]:
ruta_completa = ruta_proyecto + '/02_Datos/01_Originales/' + nombre_fichero_datos

df = pd.read_csv(ruta_completa,index_col='id',sep=';')

### Seleccionar solo las variables finales

#### Cargar la lista de variables finales

In [5]:
nombre_variables_finales = ruta_proyecto + '/05_Resultados/' + 'variables_finales.pickle'

pd.read_pickle(nombre_variables_finales).sort_index().index.to_list()

['ambito_Select',
 'descarga_lm_No',
 'ocupacion_Working Professional',
 'origen_Lead Add Form',
 'paginas_vistas_visita_mms',
 'score_actividad_mms',
 'score_perfil_mms',
 'tiempo_en_site_total_mms',
 'ult_actividad_Chat Conversation',
 'ult_actividad_Converted to Lead',
 'ult_actividad_Page Visited on Website',
 'ult_actividad_SMS Sent',
 'visitas_total_mms']

#### Apuntar (manualmente) la lista de variables finales sin extensiones

In [6]:
variables_finales = ['ambito',
                     'descarga_lm',
                     'ocupacion',
                     'origen',
                     'paginas_vistas_visita',
                     'score_actividad',
                     'score_perfil',
                     'tiempo_en_site_total',
                     'ult_actividad',
                     'visitas_total']

## ESTRUCTURA DE LOS DATASETS

### Eliminar registros

#### Por duplicados

In [7]:
df.drop_duplicates(inplace = True)

#### Por EDA

In [8]:
df= df.loc[(df.no_llamar != 'yes') & (df.no_enviar_email != 'yes') & (df.ult_actividad != 'Email Bounced')]

#### Para x

Quedarse solo con las de la lista.

In [9]:
x = df[variables_finales].copy()

#### Para y

Especificar la target.

In [10]:
target = 'compra'

Crear el y.

In [11]:
y = df[target].copy()

## CREAR EL PIPELINE

### Instanciar calidad de datos

In [12]:
x.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7229 entries, 660737 to 579533
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ambito                 6585 non-null   object 
 1   descarga_lm            7229 non-null   object 
 2   ocupacion              5409 non-null   object 
 3   origen                 7229 non-null   object 
 4   paginas_vistas_visita  7119 non-null   float64
 5   score_actividad        4068 non-null   float64
 6   score_perfil           4068 non-null   float64
 7   tiempo_en_site_total   7229 non-null   int64  
 8   ult_actividad          7139 non-null   object 
 9   visitas_total          7119 non-null   float64
dtypes: float64(4), int64(1), object(5)
memory usage: 621.2+ KB


#### Crear la funcion

In [17]:
def calidad_datos(df):
    temp = df.astype({'visitas_total': 'Int64'})    
    
    def imputar_moda(variable):
        return(variable.fillna(variable.mode()[0]))
    
    var_imputar_moda = ['ocupacion','ambito']
    
    temp[var_imputar_moda] = temp[var_imputar_moda].apply(imputar_moda)
    
    var_imputar_valor = ['origen',
                     'ult_actividad',
                     'descarga_lm']
    valor = 'desconocido'
     
    temp[var_imputar_valor] = temp[var_imputar_valor].fillna(valor)
    
    
    var_imputar_mediana = ['paginas_vistas_visita','score_actividad', 'score_perfil', 'tiempo_en_site_total','visitas_total']
    
    
    def imputar_mediana(variable):
        if pd.api.types.is_integer_dtype(variable):
            return(variable.fillna(int(variable.median())))
        else:
            return(variable.fillna(variable.median()))
    
    temp[var_imputar_mediana] = temp[var_imputar_mediana].apply(imputar_mediana)
    
    
    
    def agrupar_cat_raras(variable, criterio = 0.02):
        #Calcula las frecuencias
        frecuencias = variable.value_counts(normalize=True)
        #Identifica las que estan por debajo del criterio
        temp = [cada for cada in frecuencias.loc[frecuencias < criterio].index.values]
        #Las recodifica en 'OTROS'
        temp2 = np.where(variable.isin(temp),'OTROS',variable)
        #Devuelve el resultado
        return(temp2)
    
    
    var_imputar_cat_raras =['ocupacion','ambito','origen','ult_actividad','descarga_lm']
    
    for variable in var_imputar_cat_raras:
        temp[variable] = agrupar_cat_raras(temp[variable],criterio = 0.02)
    
    
    temp['visitas_total'] = temp['visitas_total'].clip(0,50)
    temp['paginas_vistas_visita'] = temp['paginas_vistas_visita'].clip(0,20)
    
    return(temp)

#### Convertirla en transformer

In [19]:
x.shape

(7229, 10)

In [23]:
calidad_datos(x)

,ambito,descarga_lm,ocupacion,origen,paginas_vistas_visita,score_actividad,score_perfil,tiempo_en_site_total,ult_actividad,visitas_total
id,,,,,,,,,,
660737,Select,No,Unemployed,API,0.00,15.0,15.0,0,Page Visited on Website,0
660728,Select,No,Unemployed,API,2.50,15.0,15.0,674,Email Opened,5
660727,Business Administration,Yes,Student,Landing Page Submission,2.00,14.0,20.0,1532,Email Opened,2
660719,Media and Advertising,No,Unemployed,Landing Page Submission,1.00,13.0,17.0,305,OTROS,1
660681,Select,No,Unemployed,Landing Page Submission,1.00,15.0,18.0,1428,Converted to Lead,2
...,...,...,...,...,...,...,...,...,...,...
579564,IT Projects Management,No,Unemployed,Landing Page Submission,2.67,15.0,17.0,1845,OTROS,8
579546,Media and Advertising,Yes,Unemployed,Landing Page Submission,2.00,14.0,19.0,238,SMS Sent,2
579545,Business Administration,Yes,Unemployed,Landing Page Submission,2.00,13.0,20.0,199,SMS Sent,2


In [24]:
hacer_calidad_datos = FunctionTransformer(calidad_datos)

### Instanciar transformación de variables

In [27]:
var_ohe = ['ocupacion','ambito','origen','ult_actividad','descarga_lm']
ohe = OneHotEncoder(sparse = False, handle_unknown='ignore')

var_mms =['paginas_vistas_visita','score_actividad', 'score_perfil', 'tiempo_en_site_total','visitas_total']
mms = MinMaxScaler()

### Crear el pipe del preprocesamiento

#### Crear el column transformer

In [28]:
ct = make_column_transformer(
    (ohe, var_ohe),
    (mms, var_mms),
    remainder='drop')

#### Crear el pipeline del preprocesamiento

In [29]:
pipe_prepro = make_pipeline(hacer_calidad_datos, 
                            ct)

### Instanciar el modelo

#### Instanciar el algoritmo

In [30]:
modelo = LogisticRegression(n_jobs = -1, 
                       solver = 'saga',
                       penalty = 'none',
                       C = 0.25
                       )

#### Crear el pipe final de entrenamiento

In [31]:
pipe_entrenamiento = make_pipeline(pipe_prepro,modelo)

#### Guardar el pipe final de entrenamiento

In [32]:
nombre_pipe_entrenamiento = 'pipe_entrenamiento.pickle'

ruta_pipe_entrenamiento = ruta_proyecto + '/04_Modelos/' + nombre_pipe_entrenamiento

with open(ruta_pipe_entrenamiento, mode='wb') as file:
   cloudpickle.dump(pipe_entrenamiento, file)

#### Entrenar el pipe final de ejecucion

In [33]:
pipe_ejecucion = pipe_entrenamiento.fit(x,y)

F:\cienciadedatos\envs\leadscoring\Lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
F:\cienciadedatos\envs\leadscoring\Lib\site-packages\sklearn\linear_model\_logistic.py:1183: FutureWarning: `penalty='none'`has been deprecated in 1.2 and will be removed in 1.4. To keep the past behaviour, set `penalty=None`.
  warnings.warn(
F:\cienciadedatos\envs\leadscoring\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


## GUARDAR EL PIPE

### Nombre del pipe final de ejecucion

In [34]:
nombre_pipe_ejecucion = 'pipe_ejecucion.pickle'

### Guardar el pipe final de ejecucion

In [35]:
ruta_pipe_ejecucion = ruta_proyecto + '/04_Modelos/' + nombre_pipe_ejecucion

with open(ruta_pipe_ejecucion, mode='wb') as file:
   cloudpickle.dump(pipe_ejecucion, file)